In [ ]:
using Plots
using LinearAlgebra
using Krylov
using Printf
using LaTeXStrings
using BenchmarkTools
include("poisson2d.jl")

In [ ]:
default(lw=2, markersize=6,
    xtickfont=font(12), ytickfont=font(12),
    guidefont=font(14), legendfont=font(12), titlefont=font(12))

In [ ]:
Lx = 5;
Ly = 5;
nx = 100; # (nx-1)×(ny-1) interior points
ny = 100;
Nx = nx-1;
Ny = ny-1;

x = LinRange(-Lx, Lx, nx + 1)
y = LinRange(-Ly, Ly, ny + 1)

@show Δx = x[2] - x[1];
@show Δy = y[2] - y[1];

xy = [[x_, y_] for x_ in x, y_ in y]; # mesh including boundary points
xy_int = [[x_, y_] for x_ in x[2:end-1],y_ in y[2:end-1]]; # interior mesh points

L = dirichlet_laplacian_2d_kron2(Δx, Δy, Nx, Ny);

# f(X,t) = exp(-2 *(X[1]^2 + X[2]^2)) * sin(2π*t)^2
f(X,t) = (cos(X[1])*sin(X[2]))^2 * sin(2π*t)^2
F(X,t) = vec(f.(X, t))

In [ ]:
t_vals = LinRange(0, 5, 101);
anim = @animate for t in t_vals
    z= f.(xy, t);
    if (any(isnan, z))
        error("NaN values in z")
    end
    if (any(isinf, z))
        error("Inf values in z")
    end
    contourf(x, y, z', title = @sprintf("t = %.2f", t), xlabel="x", ylabel="y", c=:viridis, 
        clims = (0, 1), levels=0:.1:1.1,colorbar=false)
    xlims!(-Lx, Lx)
    ylims!(-Ly, Ly)

end


In [ ]:
gif(anim, fps = 6)

In [ ]:
u = zeros(Nx,Ny);
U = vec(u);
u_vals = [deepcopy(u)];
t_vals = LinRange(0, 5, 101);
Δt = t_vals[2] - t_vals[1];
t = t_vals[1];
f2d = zeros(Nx,Ny);
Fvec = vec(f2d);
for t in t_vals[1:end-1]
    f2d .= f.(xy_int, t);
    U .= cg(I-Δt*L, U + Δt*Fvec)[1];
    push!(u_vals, deepcopy(u));
end    

In [ ]:
length(t_vals)

In [ ]:
anim = @animate for (t,u) in zip(t_vals, u_vals)
    contourf(x[2:end-1], y[2:end-1], u', title = @sprintf("t = %.2f", t), xlabel="x", ylabel="y", c=:viridis, 
        clims = (0, 2), colorbar=true)
    xlims!(-Lx, Lx)
    ylims!(-Ly, Ly)

end


In [ ]:
gif(anim, fps = 6)